In [ ]:
# -*- coding: utf-8 -*-
"""
3D adapted BL-PINN benchmark with strict shared-LHS error evaluation.

Mathematical structure
----------------------
Three neural-network modules are trained jointly:

1. N_phi(x,y,z) -> (phi^-, phi^+), the two outer states.
2. N_h(y,z,t) -> h, the moving transition-layer position.
3. N_Q(xi,h,y,z,t) -> (Q^-, Q^+), the two inner corrections.

The learned components satisfy the leading asymptotic subproblems for

    mu * Delta u - u_t = -u (u_x + u_y + u_z) + f(x,y,z),

where f = cos(pi*x) cos(pi*y) cos(pi*z).

Important conventions
---------------------
* The interface initial condition h(y,z,0)=0 is imposed exactly.
* The outer boundary values phi^-(-1,y,z)=-4 and phi^+(1,y,z)=2 are
  imposed exactly.
* The truncated far-field conditions Q^-(-XI_MAX)=0 and Q^+(XI_MAX)=0
  are imposed exactly.
* Matching uses only the two interface value conditions. The normal
  derivative matching condition is not added again, because it is already
  used to derive the reduced interface equation.
* Raw y,z coordinates are used. Value periodicity in y and z is imposed
  softly on all three modules phi, h, and Q.
* The normal stretched coordinate used for reconstruction is

      xi = (x-h) * sqrt(1+h_y^2+h_z^2) / mu.

  This convention recovers the explicit 3D asymptotic expression in the
  manuscript.

Benchmark protocol
------------------
* Six fixed random seeds.
* 40,000 joint Adam iterations.
* Strict reuse of the shared 13,000-point 3D DAE LHS index file.
* T_eval is averaged over 200 reconstructions after 20 warmup calls.
* Error calculation, CPU transfer, file writing, and full-field generation
  are excluded from T_eval.
* For seed 1234 only, the complete 51^4 reconstructed field is saved in
  chunks, outside all reported timings.
"""

from __future__ import annotations

import math
import os
import random
import time
from typing import Dict, List, Tuple

import numpy as np
import pandas as pd
import torch
import torch.nn as nn
import torch.nn.init as init
import torch.optim as optim
from scipy.spatial import cKDTree
from scipy.stats import qmc


# =============================================================================
# 1. Basic settings
# =============================================================================
Tensor = torch.Tensor
DEVICE = torch.device("cuda:0" if torch.cuda.is_available() else "cpu")
DTYPE = torch.float32
torch.backends.cudnn.benchmark = False

SEEDS = [33, 99, 202, 1234, 5678, 9999]
MU_LIST = [0.01]

X_MIN, X_MAX = -1.0, 1.0
Y_MIN, Y_MAX = -1.0, 1.0
Z_MIN, Z_MAX = -1.0, 1.0
T_FINAL = 0.5
L_VALUE, R_VALUE = -4.0, 2.0
H0_VALUE = 0.0

DEPTH, WIDTH = 6, 10
LEARNING_RATE = 1.0e-3
EPOCHS = 40000
XI_MAX = 12.0

# Manuscript tuple: (N_phi, N_h, N_Q, N_m, N_hp).
N_PHI = 5000
N_H = 5000
N_Q = 6000       # used independently on the left and right inner domains
N_M = 6000
N_HP = 3000      # pair count used for each periodic direction and module

# Periodic loss weights.
W_PHI_PER = 1.0
W_H_PER = 1.0
W_Q_PER = 1.0

# Honest bookkeeping of input locations appearing in one composite-loss call.
# Residual/matching locations:
#   N_phi + N_h + 2*N_Q + N_m.
# Periodic boundary locations:
#   phi: 4*N_HP (two endpoints in y and two in z),
#   h:   4*N_HP,
#   Q:   8*N_HP (two branches, two endpoints, two periodic directions).
TRAIN_INPUT_LOCATIONS_PER_ITER = (
    N_PHI + N_H + 2 * N_Q + N_M + 16 * N_HP
)

# A second count records scalar residual/constraint samples rather than network
# input locations. Each outer, inner, and matching point contributes two scalar
# conditions. The periodic count includes every scalar channel and direction.
SCALAR_CONDITIONS_PER_ITER = (
    2 * N_PHI                       # outer residuals: minus and plus
    + N_H                           # interface residual
    + 2 * N_Q                       # inner residuals: minus and plus
    + 2 * N_M                       # two value-matching equations
    + 4 * N_HP                      # phi: 2 branches x 2 directions
    + 2 * N_HP                      # h: 2 directions
    + 4 * N_HP                      # Q: 2 branches x 2 directions
)

# Strict shared-LHS test protocol.
NUM_SAMPLES = 13000
LHS_SEED = 1234
EVAL_WARMUP = 20
EVAL_REPEAT = 200
REQUIRE_SHARED_LHS_INDEX = True
BASE_PATH = "."
SAVE_LHS_PREDICTION = True
METHOD_NAME = "BLPINN"

# Full 51^4 output for plotting, seed 1234 only.
SAVE_FULL_FIELD = True
FULL_FIELD_SEED = 1234
FULL_NT = 51
FULL_NX = 51
FULL_NY = 51
FULL_NZ = 51
FULL_FIELD_CHUNK_SIZE = 50000
EXPECTED_REFERENCE_POINTS = FULL_NT * FULL_NX * FULL_NY * FULL_NZ


# =============================================================================
# 2. Utilities
# =============================================================================
def synchronize_backend() -> None:
    if DEVICE.type == "cuda":
        torch.cuda.synchronize(DEVICE)


def set_seed(seed: int) -> None:
    random.seed(seed)
    np.random.seed(seed)
    torch.manual_seed(seed)
    if DEVICE.type == "cuda":
        torch.cuda.manual_seed_all(seed)


def grad_of(
    outputs: Tensor,
    inputs: Tensor,
    *,
    create_graph: bool = True,
    retain_graph: bool = True,
) -> Tensor:
    return torch.autograd.grad(
        outputs,
        inputs,
        grad_outputs=torch.ones_like(outputs),
        create_graph=create_graph,
        retain_graph=retain_graph,
        only_inputs=True,
    )[0]


def mse(value: Tensor) -> Tensor:
    return torch.mean(value.square())


def mean_std(values) -> Tuple[float, float]:
    arr = np.asarray(values, dtype=float)
    if arr.size <= 1:
        return float(np.nanmean(arr)), 0.0
    return float(np.nanmean(arr)), float(np.nanstd(arr, ddof=1))


def compute_error(true_u, pred_u) -> Tuple[float, float]:
    true_vec = np.asarray(true_u, dtype=np.float64).reshape(-1)
    pred_vec = np.asarray(pred_u, dtype=np.float64).reshape(-1)
    if true_vec.shape != pred_vec.shape:
        raise ValueError(
            f"Shape mismatch: true={true_vec.shape}, pred={pred_vec.shape}."
        )
    denom = np.linalg.norm(true_vec)
    if not np.isfinite(denom) or denom <= 0.0:
        raise ValueError("Reference L2 norm is zero or non-finite.")
    diff = pred_vec - true_vec
    return float(np.linalg.norm(diff) / denom), float(np.max(np.abs(diff)))


def source_f(x: Tensor, y: Tensor, z: Tensor) -> Tensor:
    return (
        torch.cos(math.pi * x)
        * torch.cos(math.pi * y)
        * torch.cos(math.pi * z)
    )


def constant_like_column(value: float, reference: Tensor) -> Tensor:
    return torch.full_like(reference, float(value))


# =============================================================================
# 3. Networks and exact output transformations
# =============================================================================
class MLP(nn.Module):
    def __init__(self, in_dim: int, out_dim: int, width: int, depth: int):
        super().__init__()
        if depth < 2:
            raise ValueError("depth must be at least 2")
        layers: List[nn.Module] = [nn.Linear(in_dim, width), nn.Tanh()]
        for _ in range(depth - 2):
            layers += [nn.Linear(width, width), nn.Tanh()]
        layers.append(nn.Linear(width, out_dim))
        self.net = nn.Sequential(*layers)
        self.reset_parameters()

    def reset_parameters(self) -> None:
        for module in self.net:
            if isinstance(module, nn.Linear):
                init.xavier_normal_(module.weight)
                if module.bias is not None:
                    init.zeros_(module.bias)

    def forward(self, inputs: Tensor) -> Tensor:
        return self.net(inputs)


class AdaptedBLPINN3D(nn.Module):
    """Three jointly trained modules for phi, h, and Q."""

    def __init__(self):
        super().__init__()
        self.N_phi = MLP(3, 2, WIDTH, DEPTH)       # (x,y,z) -> two outer heads
        self.N_h = MLP(3, 1, WIDTH, DEPTH)         # (y,z,t) -> interface
        self.N_Q = MLP(5, 2, WIDTH, DEPTH)         # (xi,h,y,z,t) -> two inner heads

    def outer(self, x: Tensor, y: Tensor, z: Tensor) -> Tuple[Tensor, Tensor]:
        raw = self.N_phi(torch.cat([x, y, z], dim=1))
        phi_m = L_VALUE + (x - X_MIN) * raw[:, 0:1]
        phi_p = R_VALUE + (x - X_MAX) * raw[:, 1:2]
        return phi_m, phi_p

    def phi_m(self, x: Tensor, y: Tensor, z: Tensor) -> Tensor:
        return self.outer(x, y, z)[0]

    def phi_p(self, x: Tensor, y: Tensor, z: Tensor) -> Tensor:
        return self.outer(x, y, z)[1]

    def h(self, y: Tensor, z: Tensor, t: Tensor) -> Tensor:
        raw = self.N_h(torch.cat([y, z, t], dim=1))
        return H0_VALUE + t * raw

    def inner_raw(
        self,
        xi: Tensor,
        h: Tensor,
        y: Tensor,
        z: Tensor,
        t: Tensor,
    ) -> Tensor:
        return self.N_Q(torch.cat([xi, h, y, z, t], dim=1))

    def Q_m(
        self,
        xi: Tensor,
        h: Tensor,
        y: Tensor,
        z: Tensor,
        t: Tensor,
    ) -> Tensor:
        raw = self.inner_raw(xi, h, y, z, t)[:, 0:1]
        return ((xi + XI_MAX) / XI_MAX) * raw

    def Q_p(
        self,
        xi: Tensor,
        h: Tensor,
        y: Tensor,
        z: Tensor,
        t: Tensor,
    ) -> Tensor:
        raw = self.inner_raw(xi, h, y, z, t)[:, 1:2]
        return ((XI_MAX - xi) / XI_MAX) * raw

    def Q_both(
        self,
        xi: Tensor,
        h: Tensor,
        y: Tensor,
        z: Tensor,
        t: Tensor,
    ) -> Tuple[Tensor, Tensor]:
        raw = self.inner_raw(xi, h, y, z, t)
        q_m = ((xi + XI_MAX) / XI_MAX) * raw[:, 0:1]
        q_p = ((XI_MAX - xi) / XI_MAX) * raw[:, 1:2]
        return q_m, q_p


# =============================================================================
# 4. Fixed sampling
# =============================================================================
def sample_uniform(low: float, high: float, n: int) -> Tensor:
    return low + (high - low) * torch.rand(n, 1, device=DEVICE, dtype=DTYPE)


def sample_x(n: int) -> Tensor:
    return sample_uniform(X_MIN, X_MAX, n)


def sample_y(n: int) -> Tensor:
    return sample_uniform(Y_MIN, Y_MAX, n)


def sample_z(n: int) -> Tensor:
    return sample_uniform(Z_MIN, Z_MAX, n)


def sample_t(n: int) -> Tensor:
    return sample_uniform(0.0, T_FINAL, n)


def sample_xi_minus(n: int) -> Tensor:
    return sample_uniform(-XI_MAX, 0.0, n)


def sample_xi_plus(n: int) -> Tensor:
    return sample_uniform(0.0, XI_MAX, n)


def build_training_data() -> Dict[str, Tensor]:
    """Generate one fixed training set for one random seed."""
    return {
        # Outer residual.
        "x_phi": sample_x(N_PHI),
        "y_phi": sample_y(N_PHI),
        "z_phi": sample_z(N_PHI),
        # Interface residual.
        "y_h": sample_y(N_H),
        "z_h": sample_z(N_H),
        "t_h": sample_t(N_H),
        # Left inner residual.
        "xi_m": sample_xi_minus(N_Q),
        "y_m": sample_y(N_Q),
        "z_m": sample_z(N_Q),
        "t_m": sample_t(N_Q),
        # Right inner residual.
        "xi_p": sample_xi_plus(N_Q),
        "y_p": sample_y(N_Q),
        "z_p": sample_z(N_Q),
        "t_p": sample_t(N_Q),
        # Matching points.
        "y_match": sample_y(N_M),
        "z_match": sample_z(N_M),
        "t_match": sample_t(N_M),
        # Shared periodic pair parameters for phi.
        "x_phi_per_y": sample_x(N_HP),
        "z_phi_per_y": sample_z(N_HP),
        "x_phi_per_z": sample_x(N_HP),
        "y_phi_per_z": sample_y(N_HP),
        # Shared periodic pair parameters for h.
        "z_h_per_y": sample_z(N_HP),
        "t_h_per_y": sample_t(N_HP),
        "y_h_per_z": sample_y(N_HP),
        "t_h_per_z": sample_t(N_HP),
        # Shared periodic pair parameters for Q.
        "xi_m_q_per_y": sample_xi_minus(N_HP),
        "xi_p_q_per_y": sample_xi_plus(N_HP),
        "z_q_per_y": sample_z(N_HP),
        "t_q_per_y": sample_t(N_HP),
        "xi_m_q_per_z": sample_xi_minus(N_HP),
        "xi_p_q_per_z": sample_xi_plus(N_HP),
        "y_q_per_z": sample_y(N_HP),
        "t_q_per_z": sample_t(N_HP),
    }


# =============================================================================
# 5. Joint BL-PINN losses
# =============================================================================
def loss_outer_phi(model: AdaptedBLPINN3D, data: Dict[str, Tensor]) -> Tensor:
    x = data["x_phi"].detach().clone().requires_grad_(True)
    y = data["y_phi"].detach().clone().requires_grad_(True)
    z = data["z_phi"].detach().clone().requires_grad_(True)

    phi_m, phi_p = model.outer(x, y, z)

    pm_x = grad_of(phi_m, x)
    pm_y = grad_of(phi_m, y)
    pm_z = grad_of(phi_m, z)
    pp_x = grad_of(phi_p, x)
    pp_y = grad_of(phi_p, y)
    pp_z = grad_of(phi_p, z)

    forcing = source_f(x, y, z)
    res_m = phi_m * (pm_x + pm_y + pm_z) - forcing
    res_p = phi_p * (pp_x + pp_y + pp_z) - forcing
    loss_res = mse(res_m) + mse(res_p)

    # y-periodicity at y=-1 and y=1.
    x_y = data["x_phi_per_y"]
    z_y = data["z_phi_per_y"]
    y_min = constant_like_column(Y_MIN, x_y)
    y_max = constant_like_column(Y_MAX, x_y)
    pm_y_min, pp_y_min = model.outer(x_y, y_min, z_y)
    pm_y_max, pp_y_max = model.outer(x_y, y_max, z_y)

    # z-periodicity at z=-1 and z=1.
    x_z = data["x_phi_per_z"]
    y_z = data["y_phi_per_z"]
    z_min = constant_like_column(Z_MIN, x_z)
    z_max = constant_like_column(Z_MAX, x_z)
    pm_z_min, pp_z_min = model.outer(x_z, y_z, z_min)
    pm_z_max, pp_z_max = model.outer(x_z, y_z, z_max)

    loss_per = (
        mse(pm_y_min - pm_y_max)
        + mse(pp_y_min - pp_y_max)
        + mse(pm_z_min - pm_z_max)
        + mse(pp_z_min - pp_z_max)
    )
    return loss_res + W_PHI_PER * loss_per


def interface_quantities(
    model: AdaptedBLPINN3D,
    y_base: Tensor,
    z_base: Tensor,
    t_base: Tensor,
) -> Tuple[Tensor, Tensor, Tensor, Tensor, Tensor, Tensor, Tensor]:
    y = y_base.detach().clone().requires_grad_(True)
    z = z_base.detach().clone().requires_grad_(True)
    t = t_base.detach().clone().requires_grad_(True)

    h_val = model.h(y, z, t)
    h_y = grad_of(h_val, y)
    h_z = grad_of(h_val, z)
    h_t = grad_of(h_val, t)
    return y, z, t, h_val, h_y, h_z, h_t


def loss_interface_h(model: AdaptedBLPINN3D, data: Dict[str, Tensor]) -> Tensor:
    y, z, t, h_val, h_y, h_z, h_t = interface_quantities(
        model, data["y_h"], data["z_h"], data["t_h"]
    )
    phi_m_h, phi_p_h = model.outer(h_val, y, z)

    # h_t = 0.5*(h_y+h_z-1)*(phi^-_h+phi^+_h).
    residual = h_t - 0.5 * (h_y + h_z - 1.0) * (phi_m_h + phi_p_h)
    loss_res = mse(residual)

    # y-periodicity.
    z_y = data["z_h_per_y"]
    t_y = data["t_h_per_y"]
    y_min = constant_like_column(Y_MIN, z_y)
    y_max = constant_like_column(Y_MAX, z_y)
    h_y_min = model.h(y_min, z_y, t_y)
    h_y_max = model.h(y_max, z_y, t_y)

    # z-periodicity.
    y_z = data["y_h_per_z"]
    t_z = data["t_h_per_z"]
    z_min = constant_like_column(Z_MIN, y_z)
    z_max = constant_like_column(Z_MAX, y_z)
    h_z_min = model.h(y_z, z_min, t_z)
    h_z_max = model.h(y_z, z_max, t_z)

    loss_per = mse(h_y_min - h_y_max) + mse(h_z_min - h_z_max)
    return loss_res + W_H_PER * loss_per


def q_residual_side(
    model: AdaptedBLPINN3D,
    xi_base: Tensor,
    y_base: Tensor,
    z_base: Tensor,
    t_base: Tensor,
    side: str,
) -> Tensor:
    y, z, t, h_val, h_y, h_z, h_t = interface_quantities(
        model, y_base, z_base, t_base
    )
    phi_m_h, phi_p_h = model.outer(h_val, y, z)
    metric = torch.sqrt(1.0 + h_y.square() + h_z.square())

    xi = xi_base.detach().clone().requires_grad_(True)
    if side == "minus":
        q_val = model.Q_m(xi, h_val, y, z, t)
        phi_side = phi_m_h
    elif side == "plus":
        q_val = model.Q_p(xi, h_val, y, z, t)
        phi_side = phi_p_h
    else:
        raise ValueError(f"Unknown side: {side}")

    q_xi = grad_of(q_val, xi)
    q_xixi = grad_of(q_xi, xi)

    # General leading inner equation specialized to A(u)=-u:
    # Q_xixi + [h_t + (phi_side+Q)*(1-h_y-h_z)]/G * Q_xi = 0.
    coefficient = (
        h_t + (phi_side + q_val) * (1.0 - h_y - h_z)
    ) / metric
    return q_xixi + coefficient * q_xi


def loss_inner_Q(model: AdaptedBLPINN3D, data: Dict[str, Tensor]) -> Tensor:
    res_m = q_residual_side(
        model,
        data["xi_m"], data["y_m"], data["z_m"], data["t_m"],
        "minus",
    )
    res_p = q_residual_side(
        model,
        data["xi_p"], data["y_p"], data["z_p"], data["t_p"],
        "plus",
    )
    loss_res = mse(res_m) + mse(res_p)

    # Interface value matching only. No duplicate Q_xi matching penalty.
    y_m = data["y_match"]
    z_m = data["z_match"]
    t_m = data["t_match"]
    h_match = model.h(y_m, z_m, t_m)
    phi_m_h, phi_p_h = model.outer(h_match, y_m, z_m)
    phi_mid = 0.5 * (phi_m_h + phi_p_h)
    xi0 = torch.zeros_like(t_m)
    q0_m, q0_p = model.Q_both(xi0, h_match, y_m, z_m, t_m)
    loss_match = (
        mse(phi_m_h + q0_m - phi_mid)
        + mse(phi_p_h + q0_p - phi_mid)
    )

    # Q periodicity in y, evaluated for both inner branches.
    z_y = data["z_q_per_y"]
    t_y = data["t_q_per_y"]
    y_min = constant_like_column(Y_MIN, z_y)
    y_max = constant_like_column(Y_MAX, z_y)
    h_y_min = model.h(y_min, z_y, t_y)
    h_y_max = model.h(y_max, z_y, t_y)

    xi_m_y = data["xi_m_q_per_y"]
    xi_p_y = data["xi_p_q_per_y"]
    q_m_y_min = model.Q_m(xi_m_y, h_y_min, y_min, z_y, t_y)
    q_m_y_max = model.Q_m(xi_m_y, h_y_max, y_max, z_y, t_y)
    q_p_y_min = model.Q_p(xi_p_y, h_y_min, y_min, z_y, t_y)
    q_p_y_max = model.Q_p(xi_p_y, h_y_max, y_max, z_y, t_y)

    # Q periodicity in z, evaluated for both inner branches.
    y_z = data["y_q_per_z"]
    t_z = data["t_q_per_z"]
    z_min = constant_like_column(Z_MIN, y_z)
    z_max = constant_like_column(Z_MAX, y_z)
    h_z_min = model.h(y_z, z_min, t_z)
    h_z_max = model.h(y_z, z_max, t_z)

    xi_m_z = data["xi_m_q_per_z"]
    xi_p_z = data["xi_p_q_per_z"]
    q_m_z_min = model.Q_m(xi_m_z, h_z_min, y_z, z_min, t_z)
    q_m_z_max = model.Q_m(xi_m_z, h_z_max, y_z, z_max, t_z)
    q_p_z_min = model.Q_p(xi_p_z, h_z_min, y_z, z_min, t_z)
    q_p_z_max = model.Q_p(xi_p_z, h_z_max, y_z, z_max, t_z)

    loss_per = (
        mse(q_m_y_min - q_m_y_max)
        + mse(q_p_y_min - q_p_y_max)
        + mse(q_m_z_min - q_m_z_max)
        + mse(q_p_z_min - q_p_z_max)
    )

    return loss_res + loss_match + W_Q_PER * loss_per


def compute_composite_loss(
    model: AdaptedBLPINN3D,
    data: Dict[str, Tensor],
) -> Tuple[Tensor, Dict[str, Tensor]]:
    loss_phi = loss_outer_phi(model, data)
    loss_h = loss_interface_h(model, data)
    loss_q = loss_inner_Q(model, data)
    total = loss_phi + loss_h + loss_q
    return total, {
        "phi": loss_phi.detach(),
        "h": loss_h.detach(),
        "Q": loss_q.detach(),
    }


# =============================================================================
# 6. Joint training
# =============================================================================
def train_joint(
    model: AdaptedBLPINN3D,
    data: Dict[str, Tensor],
) -> Tuple[float, np.ndarray, Dict[str, float]]:
    optimizer = optim.Adam(model.parameters(), lr=LEARNING_RATE)
    model_parameters = tuple(model.parameters())
    loss_history_gpu: List[Tensor] = []
    last_components: Dict[str, Tensor] = {}

    model.train()
    synchronize_backend()
    start = time.perf_counter()

    for _ in range(EPOCHS):
        optimizer.zero_grad(set_to_none=True)
        total_loss, components = compute_composite_loss(model, data)
        total_loss.backward(inputs=model_parameters)
        optimizer.step()
        loss_history_gpu.append(total_loss.detach())
        last_components = components

    synchronize_backend()
    train_time = time.perf_counter() - start

    loss_history = (
        torch.stack(loss_history_gpu).cpu().numpy().astype(np.float64)
    )
    final_components = {
        key: float(value.cpu().item()) for key, value in last_components.items()
    }
    return train_time, loss_history, final_components


# =============================================================================
# 7. Reconstruction
# =============================================================================
def reconstruct_blpinn(
    model: AdaptedBLPINN3D,
    x_eval: Tensor,
    y_eval: Tensor,
    z_eval: Tensor,
    t_eval: Tensor,
    mu: float,
) -> Tensor:
    """Reconstruct U0 with xi=(x-h)*sqrt(1+h_y^2+h_z^2)/mu."""
    with torch.enable_grad():
        y_in = y_eval.detach().clone().requires_grad_(True)
        z_in = z_eval.detach().clone().requires_grad_(True)
        t_in = t_eval.detach()
        x_in = x_eval.detach()

        h_val = model.h(y_in, z_in, t_in)
        h_y, h_z = torch.autograd.grad(
            h_val.sum(),
            (y_in, z_in),
            create_graph=False,
            retain_graph=False,
        )

    # Everything below needs no coordinate derivatives during inference.
    with torch.no_grad():
        h_det = h_val.detach()
        metric = torch.sqrt(1.0 + h_y.detach().square() + h_z.detach().square())
        xi = (x_in - h_det) * metric / float(mu)

        xi_m = torch.clamp(xi, min=-XI_MAX, max=0.0)
        xi_p = torch.clamp(xi, min=0.0, max=XI_MAX)

        phi_m, phi_p = model.outer(x_in, y_eval, z_eval)
        q_m = model.Q_m(xi_m, h_det, y_eval, z_eval, t_eval)
        q_p = model.Q_p(xi_p, h_det, y_eval, z_eval, t_eval)

        return torch.where(x_in <= h_det, phi_m + q_m, phi_p + q_p)


# =============================================================================
# 8. LHS reference-data utilities
# =============================================================================
def get_target_col(df: pd.DataFrame) -> str:
    if "u" in df.columns:
        return "u"
    if "u0" in df.columns:
        return "u0"
    return str(df.columns[-1])


def load_true_solution(mu: float) -> Tuple[pd.DataFrame, str]:
    mu_id = int(round(-math.log10(mu)))
    candidates = [
        f"3d_U0_true_mu{mu:.0e}.csv",
        f"3d_U0_all_t_u_x_y_z_t_mu{mu_id}_51_mathematica_619.csv",
    ]
    if np.isclose(mu, 1.0e-2):
        candidates.append("3d_U0_all_t_u_x_y_z_t_mu2_51_mathematica_619.csv")

    for filename in candidates:
        path = os.path.join(BASE_PATH, filename)
        if os.path.exists(path):
            df = pd.read_csv(path)
            df.columns = [str(col).lower().strip() for col in df.columns]
            required = {"t", "x", "y", "z"}
            missing = required.difference(df.columns)
            if missing:
                raise ValueError(
                    f"Reference file {filename} is missing columns {sorted(missing)}."
                )
            df = df.sort_values(by=["t", "x", "y", "z"]).reset_index(drop=True)
            return df, filename

    raise FileNotFoundError(
        "Cannot find the 3D reference solution. Tried: " + ", ".join(candidates)
    )


def build_or_load_lhs_test_set(mu: float) -> Dict[str, object]:
    df_true, reference_filename = load_true_solution(mu)
    if len(df_true) != EXPECTED_REFERENCE_POINTS:
        raise ValueError(
            f"Reference file {reference_filename} has {len(df_true)} rows; "
            f"a 51^4 grid requires {EXPECTED_REFERENCE_POINTS}."
        )

    index_file = f"3d_LHS_sample_indices_mu{mu:.0e}.npy"
    sample_indices = None

    if os.path.exists(index_file):
        loaded = np.asarray(np.load(index_file), dtype=np.int64).reshape(-1)
        valid = (
            loaded.size == NUM_SAMPLES
            and np.unique(loaded).size == NUM_SAMPLES
            and loaded.min() >= 0
            and loaded.max() < len(df_true)
        )
        if valid:
            sample_indices = loaded
            print(f"[mu={mu}] Loaded shared LHS indices from {index_file}.")
        else:
            print(f"[mu={mu}] Existing shared LHS index file is invalid.")

    if sample_indices is None and REQUIRE_SHARED_LHS_INDEX:
        raise FileNotFoundError(
            f"Shared DAE LHS file {index_file} is missing or invalid. "
            "Run the 3D DAE code first, or set REQUIRE_SHARED_LHS_INDEX=False."
        )

    if sample_indices is None:
        all_points = df_true[["t", "x", "y", "z"]].to_numpy(dtype=np.float64)
        lower = all_points.min(axis=0)
        upper = all_points.max(axis=0)
        tree = cKDTree(all_points)
        selected: List[int] = []
        used = set()
        batch_id = 0

        while len(selected) < NUM_SAMPLES and batch_id < 100:
            sampler = qmc.LatinHypercube(d=4, seed=LHS_SEED + batch_id)
            points = qmc.scale(sampler.random(NUM_SAMPLES), lower, upper)
            _, candidates = tree.query(points, k=1)
            for idx in np.asarray(candidates).reshape(-1):
                idx_int = int(idx)
                if idx_int not in used:
                    used.add(idx_int)
                    selected.append(idx_int)
                    if len(selected) == NUM_SAMPLES:
                        break
            batch_id += 1

        if len(selected) < NUM_SAMPLES:
            remaining = np.setdiff1d(
                np.arange(len(df_true), dtype=np.int64),
                np.asarray(selected, dtype=np.int64),
            )
            fill = np.random.default_rng(LHS_SEED).choice(
                remaining,
                size=NUM_SAMPLES - len(selected),
                replace=False,
            )
            selected.extend(int(idx) for idx in fill)

        sample_indices = np.asarray(selected, dtype=np.int64)
        np.save(index_file, sample_indices)

    sampled = df_true.iloc[sample_indices]
    t_np = sampled["t"].to_numpy(dtype=np.float64).reshape(-1, 1)
    x_np = sampled["x"].to_numpy(dtype=np.float64).reshape(-1, 1)
    y_np = sampled["y"].to_numpy(dtype=np.float64).reshape(-1, 1)
    z_np = sampled["z"].to_numpy(dtype=np.float64).reshape(-1, 1)
    true_np = sampled[get_target_col(df_true)].to_numpy(dtype=np.float64).reshape(-1)

    print(
        f"[mu={mu}] Reference={reference_filename} | rows={len(df_true)}=51^4 | "
        f"LHS={len(sample_indices)} | index={index_file}"
    )

    return {
        "x_lhs": torch.tensor(x_np, dtype=DTYPE, device=DEVICE),
        "y_lhs": torch.tensor(y_np, dtype=DTYPE, device=DEVICE),
        "z_lhs": torch.tensor(z_np, dtype=DTYPE, device=DEVICE),
        "t_lhs": torch.tensor(t_np, dtype=DTYPE, device=DEVICE),
        "true_lhs_np": true_np,
        "x_np": x_np,
        "y_np": y_np,
        "z_np": z_np,
        "t_np": t_np,
        "n_test": int(len(sample_indices)),
        "reference_file": reference_filename,
        "lhs_index_file": index_file,
    }


# =============================================================================
# 9. Full-field reconstruction, seed 1234 only
# =============================================================================
def save_full_field_prediction(
    model: AdaptedBLPINN3D,
    mu: float,
    seed: int,
) -> str | None:
    if not SAVE_FULL_FIELD or seed != FULL_FIELD_SEED:
        return None

    total = FULL_NT * FULL_NX * FULL_NY * FULL_NZ
    t_vals = np.linspace(0.0, T_FINAL, FULL_NT, dtype=np.float64)
    x_vals = np.linspace(X_MIN, X_MAX, FULL_NX, dtype=np.float64)
    y_vals = np.linspace(Y_MIN, Y_MAX, FULL_NY, dtype=np.float64)
    z_vals = np.linspace(Z_MIN, Z_MAX, FULL_NZ, dtype=np.float64)

    filename = f"3d_{METHOD_NAME}_mu{mu:.2f}_U0_predicted_seed{seed}.csv"
    first_chunk = True
    print(f"  > Saving full 51^4 BL-PINN field: {total} rows -> {filename}")

    model.eval()
    for start in range(0, total, FULL_FIELD_CHUNK_SIZE):
        stop = min(start + FULL_FIELD_CHUNK_SIZE, total)
        flat = np.arange(start, stop, dtype=np.int64)

        iz = flat % FULL_NZ
        q = flat // FULL_NZ
        iy = q % FULL_NY
        q //= FULL_NY
        ix = q % FULL_NX
        it = q // FULL_NX

        t_np = t_vals[it]
        x_np = x_vals[ix]
        y_np = y_vals[iy]
        z_np = z_vals[iz]

        x_t = torch.tensor(x_np[:, None], dtype=DTYPE, device=DEVICE)
        y_t = torch.tensor(y_np[:, None], dtype=DTYPE, device=DEVICE)
        z_t = torch.tensor(z_np[:, None], dtype=DTYPE, device=DEVICE)
        t_t = torch.tensor(t_np[:, None], dtype=DTYPE, device=DEVICE)

        prediction = reconstruct_blpinn(model, x_t, y_t, z_t, t_t, mu)
        u_np = prediction.detach().cpu().numpy().reshape(-1)

        pd.DataFrame(
            {"u": u_np, "x": x_np, "y": y_np, "z": z_np, "t": t_np}
        ).to_csv(
            filename,
            mode="w" if first_chunk else "a",
            header=first_chunk,
            index=False,
        )
        first_chunk = False

    print(f"  > Full-field prediction saved: {filename}")
    return filename


# =============================================================================
# 10. Main benchmark
# =============================================================================
def main() -> None:
    print("\n" + "=" * 100)
    print("Starting 3D adapted BL-PINN benchmark with strict shared-LHS timing")
    print(
        f"Device={DEVICE} | dtype={DTYPE} | depth={DEPTH} | width={WIDTH} | "
        f"epochs={EPOCHS}"
    )
    print(
        "Manuscript tuple: "
        f"(N_phi,N_h,N_Q,N_m,N_hp)=({N_PHI},{N_H},{N_Q},{N_M},{N_HP})"
    )
    print(
        f"Actual network-input locations per iteration={TRAIN_INPUT_LOCATIONS_PER_ITER} | "
        f"scalar conditions per iteration={SCALAR_CONDITIONS_PER_ITER}"
    )
    print(
        f"LHS N_test={NUM_SAMPLES} | warmup={EVAL_WARMUP} | repeats={EVAL_REPEAT} | "
        f"shared-index-required={REQUIRE_SHARED_LHS_INDEX}"
    )
    print("=" * 100 + "\n")

    lhs_data = {mu: build_or_load_lhs_test_set(mu) for mu in MU_LIST}
    metrics = {mu: [] for mu in MU_LIST}

    for seed in SEEDS:
        print("\n" + "-" * 100)
        print(f"Running seed={seed}")
        print("-" * 100)
        set_seed(seed)

        training_data = build_training_data()
        model = AdaptedBLPINN3D().to(DEVICE)

        train_time, loss_history, final_components = train_joint(
            model, training_data
        )
        final_loss = float(loss_history[-1])
        per_iter_ms = train_time * 1.0e3 / EPOCHS
        per_input_location_us = (
            train_time * 1.0e6 / (EPOCHS * TRAIN_INPUT_LOCATIONS_PER_ITER)
        )
        per_scalar_condition_us = (
            train_time * 1.0e6 / (EPOCHS * SCALAR_CONDITIONS_PER_ITER)
        )

        np.save(
            f"3d_{METHOD_NAME}_loss_history_mu1e-02_seed{seed}.npy",
            loss_history,
        )
        print(
            f"  > Trained: T_train={train_time:.2f}s | e_loss={final_loss:.3e} | "
            f"components={final_components}"
        )

        model.eval()
        for parameter in model.parameters():
            parameter.requires_grad_(False)

        for mu in MU_LIST:
            data_mu = lhs_data[mu]
            x_eval = data_mu["x_lhs"]
            y_eval = data_mu["y_lhs"]
            z_eval = data_mu["z_lhs"]
            t_eval = data_mu["t_lhs"]

            # Warmup is outside T_eval.
            for _ in range(EVAL_WARMUP):
                _ = reconstruct_blpinn(
                    model, x_eval, y_eval, z_eval, t_eval, mu
                )
            synchronize_backend()

            start = time.perf_counter()
            for _ in range(EVAL_REPEAT):
                _ = reconstruct_blpinn(
                    model, x_eval, y_eval, z_eval, t_eval, mu
                )
            synchronize_backend()
            eval_time = (time.perf_counter() - start) / EVAL_REPEAT

            # Prediction transfer and errors are outside T_eval.
            prediction = reconstruct_blpinn(
                model, x_eval, y_eval, z_eval, t_eval, mu
            )
            pred_np = prediction.detach().cpu().numpy().reshape(-1)
            e2, einf = compute_error(data_mu["true_lhs_np"], pred_np)
            total_time = train_time + eval_time

            metrics[mu].append(
                {
                    "Seed": seed,
                    "N_test": data_mu["n_test"],
                    "e_loss": final_loss,
                    "loss_phi": final_components["phi"],
                    "loss_h": final_components["h"],
                    "loss_Q": final_components["Q"],
                    "e2": e2,
                    "einf": einf,
                    "T_train": train_time,
                    "T_eval": eval_time,
                    "T_total": total_time,
                    "T_train_per_iter_ms": per_iter_ms,
                    "T_train_per_iter_input_location_us": per_input_location_us,
                    "T_train_per_iter_scalar_condition_us": per_scalar_condition_us,
                    "total_trained_steps": EPOCHS,
                    "train_input_locations_per_iter": TRAIN_INPUT_LOCATIONS_PER_ITER,
                    "scalar_conditions_per_iter": SCALAR_CONDITIONS_PER_ITER,
                    "total_input_location_steps": EPOCHS * TRAIN_INPUT_LOCATIONS_PER_ITER,
                    "total_scalar_condition_steps": EPOCHS * SCALAR_CONDITIONS_PER_ITER,
                    "eval_warmup": EVAL_WARMUP,
                    "eval_repeat": EVAL_REPEAT,
                }
            )

            print(
                f"    -> mu={mu}: N_test={data_mu['n_test']} | "
                f"T_eval={eval_time:.6e}s | e2={e2:.3e} | einf={einf:.3e}"
            )

            if SAVE_LHS_PREDICTION:
                pd.DataFrame(
                    {
                        "t": data_mu["t_np"].reshape(-1),
                        "x": data_mu["x_np"].reshape(-1),
                        "y": data_mu["y_np"].reshape(-1),
                        "z": data_mu["z_np"].reshape(-1),
                        "u": pred_np,
                    }
                ).to_csv(
                    f"3d_{METHOD_NAME}_U0_predicted_LHS_mu{mu:.0e}_seed{seed}.csv",
                    index=False,
                )

            # Explicitly outside T_train, T_eval, T_total, and errors.
            save_full_field_prediction(model, mu, seed)

        del model, training_data
        if DEVICE.type == "cuda":
            torch.cuda.empty_cache()

    print("\n" + "=" * 100)
    print("ALL SEEDS COMPLETED. GENERATING SUMMARY TABLES.")
    print("=" * 100 + "\n")

    for mu in MU_LIST:
        df_mu = pd.DataFrame(metrics[mu])
        summary_file = f"3d_{METHOD_NAME}_mu{mu:.0e}_Metrics_Summary.csv"
        df_mu.to_csv(summary_file, index=False)

        columns = [
            "N_test", "e_loss", "loss_phi", "loss_h", "loss_Q", "e2", "einf",
            "T_train", "T_eval", "T_total", "T_train_per_iter_ms",
            "T_train_per_iter_input_location_us",
            "T_train_per_iter_scalar_condition_us",
            "total_trained_steps", "train_input_locations_per_iter",
            "scalar_conditions_per_iter", "total_input_location_steps",
            "total_scalar_condition_steps",
        ]
        stats = {name: mean_std(df_mu[name].values) for name in columns}

        print(f"### 3D {METHOD_NAME}, mu={mu} [Mean +/- Sample Std] ###")
        print(f"N_test: {stats['N_test'][0]:.0f} +/- {stats['N_test'][1]:.0f}")
        print(f"e_loss: {stats['e_loss'][0]:.3e} +/- {stats['e_loss'][1]:.3e}")
        print(f"loss_phi: {stats['loss_phi'][0]:.3e} +/- {stats['loss_phi'][1]:.3e}")
        print(f"loss_h: {stats['loss_h'][0]:.3e} +/- {stats['loss_h'][1]:.3e}")
        print(f"loss_Q: {stats['loss_Q'][0]:.3e} +/- {stats['loss_Q'][1]:.3e}")
        print(f"e_2: {stats['e2'][0]:.3e} +/- {stats['e2'][1]:.3e}")
        print(f"e_inf: {stats['einf'][0]:.3e} +/- {stats['einf'][1]:.3e}")
        print(f"T_train (s): {stats['T_train'][0]:.2f} +/- {stats['T_train'][1]:.2f}")
        print(f"T_eval (s): {stats['T_eval'][0]:.6e} +/- {stats['T_eval'][1]:.6e}")
        print(f"T_total (s): {stats['T_total'][0]:.2f} +/- {stats['T_total'][1]:.2f}")
        print(
            "T_train/iter (ms): "
            f"{stats['T_train_per_iter_ms'][0]:.4f} +/- "
            f"{stats['T_train_per_iter_ms'][1]:.4f}"
        )
        print(
            "T_train/(iter*input-location) (us): "
            f"{stats['T_train_per_iter_input_location_us'][0]:.4f} +/- "
            f"{stats['T_train_per_iter_input_location_us'][1]:.4f}"
        )
        print(
            "T_train/(iter*scalar-condition) (us): "
            f"{stats['T_train_per_iter_scalar_condition_us'][0]:.4f} +/- "
            f"{stats['T_train_per_iter_scalar_condition_us'][1]:.4f}"
        )
        print(f"Summary saved: {summary_file}\n")


if __name__ == "__main__":
    main()



Starting 3D adapted BL-PINN benchmark with strict shared-LHS timing
Device=cuda:0 | dtype=torch.float32 | depth=6 | width=10 | epochs=40000
Manuscript tuple: (N_phi,N_h,N_Q,N_m,N_hp)=(5000,5000,6000,6000,3000)
Actual network-input locations per iteration=76000 | scalar conditions per iteration=69000
LHS N_test=13000 | warmup=20 | repeats=200 | shared-index-required=True

[mu=0.01] Loaded shared LHS indices from 3d_LHS_sample_indices_mu1e-02.npy.
[mu=0.01] Reference=3d_U0_all_t_u_x_y_z_t_mu2_51_mathematica_619.csv | rows=6765201=51^4 | LHS=13000 | index=3d_LHS_sample_indices_mu1e-02.npy

----------------------------------------------------------------------------------------------------
Running seed=33
----------------------------------------------------------------------------------------------------
